# Sprint 5 · Land Intelligence (notebook starter)

**Colab workflow**

1. Set `RUN_IN_COLAB = True` in the Paths cell
2. Upload your earlier Colab exports
3. Run merge → save `land-intelligence.json` → map / search
4. Download the JSON into the Leaflet web app

| Typical upload | From |
| --- | --- |
| `sprint1_selected_towns.geojson` | Sprint 1 selected towns GeoJSON |
| `market_dataset.csv` | Sprint 2 (optional) |
| `market_intelligence.csv` | Sprint 3 Colab export |
| `development_change.csv` | Sprint 4 Colab export |

`RUN_IN_COLAB = False` is only if you are running this notebook on your computer inside the GeoRAD starter folder.


In [ ]:
%pip install -q pandas folium


## 0. Paths (edit this if your filenames differ)


In [ ]:
from pathlib import Path

# ------------------------------------------------------------
# WHERE ARE YOUR FILES?
# ------------------------------------------------------------
# Colab: upload files with the next cell (or the Files sidebar),
# then keep RUN_IN_COLAB = True and set the filenames below.
#
# Local (inside the GeoRAD starter folder): set RUN_IN_COLAB = False.
# That uses the sample/previous-sprint files shipped with the starter.
# ------------------------------------------------------------

RUN_IN_COLAB = True

if RUN_IN_COLAB:
    # Filenames as they appear in Colab after upload (usually /content/...)
    SPRINT1_TOWNS = Path("sprint1_selected_towns.geojson")
    SPRINT2_MARKET = Path("market_dataset.csv")  # optional
    SPRINT3_MARKET = Path("market_intelligence.csv")
    SPRINT4_CHANGE = Path("development_change.csv")
    OUTPUT_JSON = Path("land-intelligence.json")
else:
    # Only for local use next to mentorship-starters/land-intelligence/
    local_dir = Path("../data/previous-sprints")
    SPRINT1_TOWNS = local_dir / "sprint1_selected_towns.geojson"
    SPRINT2_MARKET = local_dir / "sprint2_market_dataset.csv"
    SPRINT3_MARKET = local_dir / "sprint3_market_intelligence.csv"
    SPRINT4_CHANGE = local_dir / "sprint4_development_change.csv"
    OUTPUT_JSON = Path("../data/land-intelligence.json")

print("RUN_IN_COLAB =", RUN_IN_COLAB)
for label, path in [
    ("Sprint 1 towns GeoJSON", SPRINT1_TOWNS),
    ("Sprint 2 market dataset (optional)", SPRINT2_MARKET),
    ("Sprint 3 market intelligence", SPRINT3_MARKET),
    ("Sprint 4 development change", SPRINT4_CHANGE),
    ("Output JSON for web app", OUTPUT_JSON),
]:
    print(f"{label}: {path} | exists={path.exists()}")


## 0b. Upload files (Colab)


In [ ]:
# Colab only: pick the files from your computer.
# Rename is not required if your filenames already match the Paths cell.
# If your filenames differ, either rename here or edit the Paths cell.

if RUN_IN_COLAB:
    from google.colab import files

    print("Upload Sprint 1 towns GeoJSON, Sprint 3 market intelligence, and Sprint 4 development change.")
    print("Sprint 2 market dataset is optional.")
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
else:
    print("Local mode: skipping upload. Using files under ../data/previous-sprints/")


## 0c. Helpers


In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
import folium
from IPython.display import display, Markdown


def join_key(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower()


def parse_naira_number(value):
    """Pull a numeric amount from Colab strings like '₦106,688 / sqm'."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value)
    match = re.search(r"([\d,]+(?:\.\d+)?)", text.replace(" ", ""))
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None


def format_naira(value) -> str:
    number = parse_naira_number(value)
    if number is None:
        return "-" if value in (None, "") else str(value)
    return f"₦{int(number):,}"


def _ring_centroid(ring):
    xs = [pt[0] for pt in ring]
    ys = [pt[1] for pt in ring]
    return sum(xs) / len(xs), sum(ys) / len(ys)


def geometry_centroid(geometry: dict):
    gtype = geometry.get("type")
    coords = geometry.get("coordinates")
    if gtype == "Point":
        lng, lat = coords
        return float(lat), float(lng)
    if gtype == "MultiPoint":
        lng, lat = coords[0]
        return float(lat), float(lng)
    if gtype == "Polygon":
        lng, lat = _ring_centroid(coords[0])
        return float(lat), float(lng)
    if gtype == "MultiPolygon":
        lng, lat = _ring_centroid(coords[0][0])
        return float(lat), float(lng)
    raise ValueError(f"Unsupported geometry type: {gtype}")


NAME_KEYS = [
    "Town", "town", "TOWN",
    "name", "Name", "NAME",
    "LOCATION", "Location", "location",
    "ward", "Ward", "WARD",
    "community", "Community", "COMMUNITY",
    "place", "Place", "label", "Label", "title", "Title",
    "town_name", "Town_Name", "TownName", "admin_name", "ADMIN_NAME",
    "NAME_1", "NAME_2", "NAMELSAD",
]


def pick_town_name(feature: dict):
    props = feature.get("properties")
    if not isinstance(props, dict):
        props = {}

    for key in NAME_KEYS:
        val = props.get(key)
        if val is None:
            continue
        text = str(val).strip()
        if text and text.lower() not in {"none", "null", "nan"}:
            return text

    skip = {"description", "desc", "id", "fid", "objectid", "gid", "shape_leng", "shape_area"}
    for key, val in props.items():
        if not isinstance(val, (str, int, float)):
            continue
        if str(key).lower() in skip:
            continue
        text = str(val).strip()
        if text and text.lower() not in {"none", "null", "nan"}:
            return text

    fid = feature.get("id")
    if fid is not None and str(fid).strip():
        return str(fid).strip()
    return None


def iter_geojson_features(payload):
    if isinstance(payload, list):
        return payload
    if not isinstance(payload, dict):
        raise ValueError("GeoJSON must be a FeatureCollection, Feature, or list of features.")
    gtype = payload.get("type")
    if gtype == "FeatureCollection":
        return payload.get("features") or []
    if gtype == "Feature":
        return [payload]
    raise ValueError(f"Unsupported GeoJSON type: {gtype}")


def load_sprint1_towns(path=None) -> pd.DataFrame:
    path = Path(path or SPRINT1_TOWNS)
    if not path.exists():
        raise FileNotFoundError(
            f"Sprint 1 GeoJSON not found: {path.resolve()}. "
            "Upload your selected-towns GeoJSON and check the Paths cell filename."
        )

    payload = json.loads(path.read_text(encoding="utf-8-sig"))
    features = iter_geojson_features(payload)

    rows = []
    property_keys = []
    for feature in features:
        if not isinstance(feature, dict):
            continue
        props = feature.get("properties") or {}
        if isinstance(props, dict) and props and not property_keys:
            property_keys = list(props.keys())
        town = pick_town_name(feature)
        geometry = feature.get("geometry")
        if not town or not geometry:
            continue
        lat, lng = geometry_centroid(geometry)
        rows.append({"town": town, "lat": lat, "lng": lng})

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(
            "No towns found in Sprint 1 GeoJSON. "
            f"Features seen: {len(features)}. Property keys seen: {property_keys or '(none)'}. "
            "Each feature needs a town name property (name / Town / NAME / ward / community) "
            "and a Point or Polygon geometry. "
            "If your file is the LCDA boundary only, upload the selected-towns GeoJSON instead."
        )
    return df


def merge_previous_sprints(
    towns_path=None,
    market2_path=None,
    market3_path=None,
    change_path=None,
) -> pd.DataFrame:
    """Combine Sprint 1 GeoJSON + Sprint 3 (+ optional Sprint 2) + Sprint 4."""
    towns_path = Path(towns_path or SPRINT1_TOWNS)
    market2_path = Path(market2_path or SPRINT2_MARKET)
    market3_path = Path(market3_path or SPRINT3_MARKET)
    change_path = Path(change_path or SPRINT4_CHANGE)

    towns = load_sprint1_towns(towns_path)
    market = pd.read_csv(market3_path)
    change = pd.read_csv(change_path)

    market = market.rename(
        columns={
            "Town": "town",
            "Current Market Range": "currentMarketRange",
            "Indicative Market Level": "indicativeMarketLevel",
            "Current Land Price/sqm": "currentLandPricePerSqm",
            "Supporting Source": "supportingSource",
            "Date Searched": "dateSearched",
        }
    )

    if market2_path.exists():
        market2 = pd.read_csv(market2_path).rename(
            columns={
                "Town": "town",
                "Current Land Price/sqm": "currentLandPricePerSqm",
                "Supporting Source": "supportingSource",
                "Date Searched": "dateSearched",
            }
        )
        market2["join_key"] = join_key(market2["town"])
        market["join_key"] = join_key(market["town"])
        keep = ["join_key", "currentLandPricePerSqm", "supportingSource", "dateSearched"]
        market = market.merge(market2[keep], on="join_key", how="left", suffixes=("", "_s2"))
        for col in ("currentLandPricePerSqm", "supportingSource", "dateSearched"):
            alt = f"{col}_s2"
            if alt in market.columns:
                market[col] = market[col].combine_first(market[alt])
                market = market.drop(columns=[alt])
        market = market.drop(columns=["join_key"])

    change = change.rename(
        columns={
            "Location": "town",
            "Earlier Year": "earlierYear",
            "Earlier Development": "earlierDevelopment",
            "Recent Year": "recentYear",
            "Recent Development": "recentDevelopment",
            "Development Change": "developmentChange",
            "Development Change Level": "developmentChangeLevel",
            "Notes": "notes",
        }
    )

    for frame in (towns, market, change):
        frame["join_key"] = join_key(frame["town"])

    merged = (
        market.merge(change.drop(columns=["town"]), on="join_key", how="inner")
        .merge(towns.drop(columns=["town"]), on="join_key", how="inner")
        .drop(columns=["join_key"])
    )

    if len(merged) == 0:
        raise ValueError(
            "Merge returned 0 rows. Check that Town / Location / GeoJSON name values match."
        )

    if "indicativeMarketLevel" in merged.columns:
        merged["indicativeMarketLevelValue"] = merged["indicativeMarketLevel"].map(
            parse_naira_number
        )
    if "currentLandPricePerSqm" in merged.columns:
        merged["currentLandPricePerSqmValue"] = merged["currentLandPricePerSqm"].map(
            parse_naira_number
        )

    preferred = [
        "town",
        "lat",
        "lng",
        "currentMarketRange",
        "indicativeMarketLevel",
        "indicativeMarketLevelValue",
        "currentLandPricePerSqm",
        "currentLandPricePerSqmValue",
        "supportingSource",
        "dateSearched",
        "earlierYear",
        "earlierDevelopment",
        "recentYear",
        "recentDevelopment",
        "developmentChange",
        "developmentChangeLevel",
        "notes",
    ]
    cols = [c for c in preferred if c in merged.columns]
    return merged[cols]


def save_land_intelligence(df: pd.DataFrame, path=None) -> Path:
    path = Path(path or OUTPUT_JSON)
    records = json.loads(df.to_json(orient="records"))
    for row in records:
        if row.get("indicativeMarketLevelValue") is not None:
            row["indicativeMarketLevel"] = row["indicativeMarketLevelValue"]
        if row.get("currentLandPricePerSqmValue") is not None:
            row["currentLandPricePerSqm"] = row["currentLandPricePerSqmValue"]
        row.pop("indicativeMarketLevelValue", None)
        row.pop("currentLandPricePerSqmValue", None)
    path.write_text(json.dumps(records, indent=2), encoding="utf-8")
    return path


def search_town(df: pd.DataFrame, query: str):
    query_norm = " ".join(str(query).strip().lower().split())
    if not query_norm:
        return None, "Enter a town or ward name to search."

    match = df[join_key(df["town"]) == query_norm]
    if match.empty:
        return None, "Town not available."
    return match.iloc[0], f"Found {match.iloc[0]['town']}."


def make_map(df: pd.DataFrame, highlight_town=None) -> folium.Map:
    center = [df["lat"].mean(), df["lng"].mean()]
    # Same Esri street basemap as Sprint 3 / 4 Colabs (no API key).
    BASEMAP_TILES = (
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Street_Map/MapServer/tile/{z}/{y}/{x}"
    )
    fmap = folium.Map(
        location=center,
        zoom_start=11,
        tiles=BASEMAP_TILES,
        attr="Tiles &copy; Esri",
    )

    for _, row in df.iterrows():
        is_highlight = (
            highlight_town is not None
            and str(row["town"]).strip().lower() == highlight_town.strip().lower()
        )
        level = row.get("indicativeMarketLevelValue", row.get("indicativeMarketLevel"))
        popup = folium.Popup(
            (
                f"<b>{row['town']}</b><br>"
                f"Market: {format_naira(level)} / sqm<br>"
                f"Change: {row.get('developmentChangeLevel') or '-'}"
            ),
            max_width=280,
        )
        folium.CircleMarker(
            location=[row["lat"], row["lng"]],
            radius=10 if is_highlight else 7,
            color="#0f766e" if is_highlight else "#0b1f33",
            fill=True,
            fill_opacity=0.85,
            popup=popup,
            tooltip=row["town"],
        ).add_to(fmap)

    return fmap


def show_intelligence(row: pd.Series):
    market_range = row.get("currentMarketRange") or "-"
    display(
        Markdown(
            f"""
### {row['town']}

| Signal | Value |
| --- | --- |
| Current market range | {market_range} |
| Indicative market level | {format_naira(row.get('indicativeMarketLevelValue', row.get('indicativeMarketLevel')))} / sqm |
| Current land price / sqm | {format_naira(row.get('currentLandPricePerSqmValue', row.get('currentLandPricePerSqm')))} |
| Supporting source | {row.get('supportingSource') or '-'} |
| Date searched | {row.get('dateSearched') or '-'} |
| Development {row.get('earlierYear')} | {row.get('earlierDevelopment')} |
| Development {row.get('recentYear')} | {row.get('recentDevelopment')} |
| Development change | {row.get('developmentChange')} |
| Development change level | {row.get('developmentChangeLevel') or '-'} |
| Notes | {row.get('notes') or '-'} |
"""
        )
    )


print("Helpers ready.")
print("SPRINT1_TOWNS =", SPRINT1_TOWNS)
print("SPRINT3_MARKET =", SPRINT3_MARKET)
print("SPRINT4_CHANGE =", SPRINT4_CHANGE)
print("OUTPUT_JSON =", OUTPUT_JSON)


## 0d. Peek at Sprint 1 GeoJSON properties

If town loading fails, run this to see the property keys in your file.


In [ ]:
path = Path(SPRINT1_TOWNS)
print("File:", path.resolve(), "| exists=", path.exists())
if path.exists():
    payload = json.loads(path.read_text(encoding="utf-8-sig"))
    features = iter_geojson_features(payload)
    print("Type:", payload.get("type") if isinstance(payload, dict) else type(payload))
    print("Feature count:", len(features))
    if features:
        props = features[0].get("properties")
        print("First feature property keys:", list(props.keys()) if isinstance(props, dict) else props)
        print("First feature properties:", props)
        print("First geometry type:", (features[0].get("geometry") or {}).get("type"))


## 1. Inspect previous Colab outputs

Confirm your five towns appear in each table.


In [ ]:
market = pd.read_csv(SPRINT3_MARKET)
change = pd.read_csv(SPRINT4_CHANGE)
towns = load_sprint1_towns(SPRINT1_TOWNS)

print("Sprint 3 market intelligence")
display(market)
print("Sprint 4 development change")
display(change)
print("Sprint 1 selected towns (from GeoJSON)")
display(towns)
if SPRINT2_MARKET.exists():
    print("Sprint 2 market dataset")
    display(pd.read_csv(SPRINT2_MARKET))
else:
    print("Sprint 2 file not found (optional). Continuing without it.")


## 2. Merge into one Land Intelligence database

Join key: normalized town name across Sprint 3 `Town`, Sprint 4 `Location`, and Sprint 1 GeoJSON `Town` / `name`.


In [ ]:
df = merge_previous_sprints()
print(f"Integrated {len(df)} locations")
df


## 3. Save for the web app


In [ ]:
saved = save_land_intelligence(df)
print(f"Saved {saved.resolve()}")
print(saved.read_text(encoding="utf-8")[:500], "...")


## 4. Map all locations


In [ ]:
make_map(df)


## 5. Search a town

Change `QUERY` to one of your towns. Try an unknown name to confirm the unavailable message.


In [ ]:
QUERY = "Ikeja GRA"  # try one of your towns, then try "Nowhere"

row, message = search_town(df, QUERY)
print(message)

if row is not None:
    show_intelligence(row)
    display(make_map(df, highlight_town=str(row["town"])))


## Your turn

1. Upload your real Sprint 1 / 3 / 4 exports (and Sprint 2 if you have it).
2. Re-run from Paths → merge → save.
3. Download `land-intelligence.json` from the Colab Files panel.
4. Put it in the web app as `data/land-intelligence.json`, then run the HTML app.
